# Flights Analytics Notebook

Este notebook cubre:

1. Preguntas **P1–P5** sobre PostgreSQL (Read Replica)
2. Preguntas **W1–W3** (`W2` en Athena)
3. Regresión lineal con `statsmodels`
4. Pronóstico con `StatsForecast`

> **Antes de correrlo**:
> - Bronze, Silver y Gold deben existir en Athena
> - PostgreSQL debe estar cargado
> - debes tener el endpoint de la Read Replica

## 0. Dependencias

Si hace falta, instala dependencias desde el proyecto con `uv`:

```bash
uv sync
uv run jupyter lab
```

o, si prefieres:

```bash
uv add -r requirements.txt
uv run jupyter lab
```

In [ ]:
# %pip install -r requirements.txt

## 1. Imports y configuración general

In [ ]:
import os
import json
import getpass

import boto3
import pandas as pd
import awswrangler as wr
import matplotlib.pyplot as plt
from sqlalchemy import create_engine

plt.style.use("ggplot")
pd.set_option("display.max_columns", 100)

## 2. Parámetros del proyecto

### Cómo llenar estas variables

- `REPLICA_ENDPOINT`: endpoint de la Read Replica
- `BUCKET`: bucket donde escribiste Bronze/Silver/Gold
- `ATHENA_OUTPUT`: carpeta para resultados de Athena
- `REGION`: normalmente `us-east-1`

### Credenciales PostgreSQL

Tienes dos opciones:

1. **Segura / recomendada:** usar `Secrets Manager`
2. **Manual:** usar usuario y password compartidos por tu equipo

El notebook soporta ambas.

In [ ]:
REGION = os.getenv("AWS_REGION", "us-east-1")
REPLICA_ENDPOINT = os.getenv(
    "FLIGHTS_REPLICA_ENDPOINT",
    "REEMPLAZA_AQUI_TU_RDS_REPLICA_ENDPOINT",
)
BUCKET = os.getenv("FLIGHTS_BUCKET", "REEMPLAZA_AQUI_TU_BUCKET")
ATHENA_OUTPUT = os.getenv("FLIGHTS_ATHENA_OUTPUT", f"s3://{BUCKET}/athena-results/")

USE_SECRETS_MANAGER = False
SECRET_ID = os.getenv("FLIGHTS_SECRET_ID", "itam/rds/northwind/credentials")

DB_USER = os.getenv("FLIGHTS_DB_USER", "itam")
DB_NAME = os.getenv("FLIGHTS_DB_NAME", "flights")
DB_PORT = int(os.getenv("FLIGHTS_DB_PORT", "5432"))
DB_PASSWORD = os.getenv("FLIGHTS_DB_PASSWORD")

if not USE_SECRETS_MANAGER and not DB_PASSWORD:
    try:
        DB_PASSWORD = getpass.getpass("Password PostgreSQL: ")
    except Exception:
        DB_PASSWORD = "REEMPLAZA_AQUI_TU_PASSWORD"

## 3. Construir conexiones

In [ ]:
def get_pg_creds():
    if USE_SECRETS_MANAGER:
        client = boto3.client("secretsmanager", region_name=REGION)
        secret = client.get_secret_value(SecretId=SECRET_ID)
        payload = json.loads(secret["SecretString"])
        return {
            "username": payload["username"],
            "password": payload["password"],
            "dbname": payload["dbname"],
            "port": int(payload["port"]),
        }
    return {
        "username": DB_USER,
        "password": DB_PASSWORD,
        "dbname": DB_NAME,
        "port": DB_PORT,
    }


pg_creds = get_pg_creds()

engine_replica = create_engine(
    f"postgresql+psycopg2://{pg_creds['username']}:{pg_creds['password']}"
    f"@{REPLICA_ENDPOINT}:{pg_creds['port']}/{pg_creds['dbname']}"
)

def run_pg_query(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, engine_replica)

def run_athena_query(sql: str, database: str) -> pd.DataFrame:
    return wr.athena.read_sql_query(
        sql=sql,
        database=database,
        ctas_approach=False,
        s3_output=ATHENA_OUTPUT,
    )

## 4. Helpers para gráficas

In [ ]:
def plot_bar(df: pd.DataFrame, x: str, y: str, title: str, rotation: int = 45):
    ax = df.plot(kind="bar", x=x, y=y, legend=False, figsize=(10, 4), title=title)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    plt.xticks(rotation=rotation)
    plt.tight_layout()
    plt.show()

def plot_line(df: pd.DataFrame, x: str, y: str, title: str):
    ax = df.plot(kind="line", x=x, y=y, marker="o", legend=False, figsize=(10, 4), title=title)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    plt.tight_layout()
    plt.show()

## 5. Validaciones iniciales

In [ ]:
sql_counts = '''SELECT 'airlines' AS table_name, COUNT(*) AS total FROM airlines
UNION ALL
SELECT 'airports', COUNT(*) FROM airports
UNION ALL
SELECT 'flights', COUNT(*) FROM flights;'''
df_counts = run_pg_query(sql_counts)
df_counts

In [ ]:
sql_w3_columns = '''SELECT column_name
FROM information_schema.columns
WHERE table_schema = 'public'
  AND table_name = 'flights'
  AND column_name IN ('flight_number', 'scheduled_departure');'''
df_w3_columns = run_pg_query(sql_w3_columns)
df_w3_columns

## 6. Preguntas P1–P5 (PostgreSQL)

In [ ]:
P1_SQL = '''SELECT
    origin_airport,
    destination_airport,
    COUNT(*) AS total_flights
FROM flights
GROUP BY origin_airport, destination_airport
ORDER BY total_flights DESC, origin_airport, destination_airport
LIMIT 10;'''
DF_P1 = run_pg_query(P1_SQL)
DF_P1

In [ ]:
P2_SQL = '''SELECT
    airline,
    COUNT(*) AS total_flights,
    SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END) AS total_cancelled,
    ROUND(
        100.0 * SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END)::numeric / COUNT(*),
        2
    ) AS cancelled_pct
FROM flights
GROUP BY airline
ORDER BY cancelled_pct DESC, total_flights DESC
LIMIT 5;'''
DF_P2 = run_pg_query(P2_SQL)
DF_P2

In [ ]:
P3_SQL = '''SELECT
    cancellation_reason,
    COUNT(*) AS total_cancelled
FROM flights
WHERE cancellation_reason IS NOT NULL
GROUP BY cancellation_reason
ORDER BY total_cancelled DESC;'''
DF_P3 = run_pg_query(P3_SQL)
DF_P3

In [ ]:
P4_SQL = '''SELECT
    month,
    ROUND(AVG(departure_delay)::numeric, 2) AS avg_departure_delay
FROM flights
WHERE cancelled = 0
  AND departure_delay > 0
GROUP BY month
ORDER BY month;'''
DF_P4 = run_pg_query(P4_SQL)
DF_P4

In [ ]:
P5_SQL = '''SELECT
    origin_airport,
    ROUND(SUM(weather_delay)::numeric, 2) AS total_weather_delay_minutes
FROM flights
GROUP BY origin_airport
ORDER BY total_weather_delay_minutes DESC NULLS LAST
LIMIT 10;'''
DF_P5 = run_pg_query(P5_SQL)
DF_P5

### Visualizaciones P1–P5

In [ ]:
DF_P1["route"] = DF_P1["origin_airport"] + " → " + DF_P1["destination_airport"]
plot_bar(DF_P1, "route", "total_flights", "P1. Top 10 rutas por número de vuelos", rotation=70)

In [ ]:
plot_bar(DF_P2, "airline", "cancelled_pct", "P2. Top 5 aerolíneas por porcentaje de cancelación", rotation=0)

In [ ]:
plot_bar(DF_P3, "cancellation_reason", "total_cancelled", "P3. Cancelaciones por causa", rotation=0)

In [ ]:
plot_line(DF_P4, "month", "avg_departure_delay", "P4. Retraso promedio de salida por mes")

In [ ]:
plot_bar(DF_P5, "origin_airport", "total_weather_delay_minutes", "P5. Minutos totales de weather delay", rotation=70)

## 7. Window Functions

### W1 — PostgreSQL

In [ ]:
W1_SQL = '''WITH ranked AS (
    SELECT
        airline,
        origin_airport,
        destination_airport,
        arrival_delay,
        RANK() OVER (
            PARTITION BY airline
            ORDER BY arrival_delay DESC NULLS LAST
        ) AS rnk
    FROM flights
)
SELECT
    airline,
    origin_airport,
    destination_airport,
    arrival_delay
FROM ranked
WHERE rnk = 1
ORDER BY arrival_delay DESC NULLS LAST, airline;'''
DF_W1 = run_pg_query(W1_SQL)
DF_W1

In [ ]:
plot_bar(DF_W1.sort_values("arrival_delay", ascending=False), "airline", "arrival_delay", "W1. Máximo arrival delay por aerolínea", rotation=70)

### W2 — Athena

> Recuerda: esta pregunta se hace sobre `flights_silver.flights_monthly`, no sobre PostgreSQL.

In [ ]:
W2_SQL = '''WITH monthly AS (
    SELECT
        month,
        SUM(total_flights) AS total_flights
    FROM flights_silver.flights_monthly
    GROUP BY month
)
SELECT
    month,
    total_flights,
    LAG(total_flights) OVER (ORDER BY month) AS prev_month_flights,
    total_flights - LAG(total_flights) OVER (ORDER BY month) AS diff_abs,
    ROUND(
        100.0 * (total_flights - LAG(total_flights) OVER (ORDER BY month))
        / NULLIF(LAG(total_flights) OVER (ORDER BY month), 0),
        2
    ) AS diff_pct
FROM monthly
ORDER BY month;'''
DF_W2 = run_athena_query(W2_SQL, 'flights_silver')
DF_W2

In [ ]:
plot_line(DF_W2, "month", "total_flights", "W2. Total de vuelos por mes")

### W3 — PostgreSQL

In [ ]:
W3_SQL = '''WITH ranked AS (
    SELECT
        year,
        month,
        day,
        origin_airport,
        flight_number,
        airline,
        destination_airport,
        scheduled_departure,
        ROW_NUMBER() OVER (
            PARTITION BY year, month, day, origin_airport
            ORDER BY scheduled_departure ASC
        ) AS rn
    FROM flights
    WHERE year = 2015
      AND month = 1
      AND day = 1
      AND origin_airport = 'LAX'
      AND scheduled_departure IS NOT NULL
)
SELECT
    flight_number,
    airline,
    destination_airport,
    LPAD(scheduled_departure::text, 4, '0') AS scheduled_departure
FROM ranked
WHERE rn <= 5
ORDER BY rn;'''
DF_W3 = run_pg_query(W3_SQL)
DF_W3

In [ ]:
DF_W3_plot = DF_W3.copy()
DF_W3_plot["scheduled_departure_num"] = DF_W3_plot["scheduled_departure"].astype(int)
plot_bar(DF_W3_plot, "flight_number", "scheduled_departure_num", "W3. Primeros 5 vuelos de LAX el 2015-01-01", rotation=0)

## 8. Regresión lineal (8.1)

Objetivo:
- explicar `arrival_delay`
- no optimizar un predictor de producción

In [ ]:
import numpy as np
import statsmodels.api as sm
from statsmodels.graphics.gofplots import qqplot
from sklearn.metrics import mean_squared_error

In [ ]:
REG_SQL = '''SELECT
    arrival_delay,
    departure_delay,
    distance,
    air_system_delay,
    airline_delay,
    weather_delay,
    late_aircraft_delay,
    security_delay
FROM flights_gold.vuelos_analitica
WHERE cancelled = 0
  AND arrival_delay IS NOT NULL'''
df_reg = run_athena_query(REG_SQL, 'flights_gold').dropna()
df_reg.head()

In [ ]:
y = df_reg["arrival_delay"]

X = df_reg[
    [
        "departure_delay",
        "distance",
        "air_system_delay",
        "airline_delay",
        "weather_delay",
        "late_aircraft_delay",
        "security_delay",
    ]
]

X_const = sm.add_constant(X)
ols_model = sm.OLS(y, X_const).fit()

print(ols_model.summary())

In [ ]:
y_hat = ols_model.predict(X_const)
residuals = y - y_hat

r2 = ols_model.rsquared
rmse = np.sqrt(mean_squared_error(y, y_hat))

print("R²:", round(r2, 4))
print("RMSE:", round(rmse, 4))

In [ ]:
coef = ols_model.params.drop("const")
ci = ols_model.conf_int().loc[coef.index]

coef_df = pd.DataFrame(
    {
        "feature": coef.index,
        "coef": coef.values,
        "ci_low": ci[0].values,
        "ci_high": ci[1].values,
    }
).sort_values("coef")

plt.figure(figsize=(8, 5))
plt.barh(coef_df["feature"], coef_df["coef"])
plt.errorbar(
    coef_df["coef"],
    coef_df["feature"],
    xerr=[
        coef_df["coef"] - coef_df["ci_low"],
        coef_df["ci_high"] - coef_df["coef"],
    ],
    fmt="none",
    capsize=4,
)
plt.axvline(0, linestyle="--")
plt.title("Coeficientes OLS con IC 95%")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_hat, y, alpha=0.1)
lims = [min(y_hat.min(), y.min()), max(y_hat.max(), y.max())]
plt.plot(lims, lims, linestyle="--")
plt.xlabel("ŷ (predicho)")
plt.ylabel("y (real)")
plt.title("Predichos vs reales")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(y_hat, residuals, alpha=0.1)
plt.axhline(0, linestyle="--")
plt.xlabel("ŷ (predicho)")
plt.ylabel("Residuo")
plt.title("Residuos vs predichos")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 6))
qqplot(residuals, line="45", fit=True)
plt.title("Q-Q plot de residuos")
plt.tight_layout()
plt.show()

### Interpretación sugerida

Escribe aquí 3–5 oraciones sobre:

- qué coeficiente parece más importante
- qué tan alineados están los puntos en `ŷ` vs `y`
- si en residuos vs predichos aparece patrón
- si el Q-Q plot muestra colas alejadas de la normal

## 9. Forecast de series de tiempo (8.2)

In [ ]:
# %pip install statsforecast
from statsforecast import StatsForecast
from statsforecast.models import AutoETS, AutoARIMA, AutoTheta
from statsforecast.arima import arima_string
from sklearn.metrics import mean_absolute_error

In [ ]:
TS_SQL = '''SELECT
    month,
    SUM(total_flights) AS total_flights
FROM flights_silver.flights_monthly
GROUP BY month
ORDER BY month'''
df_month = run_athena_query(TS_SQL, 'flights_silver')
df_month

In [ ]:
df_ts = pd.DataFrame(
    {
        "unique_id": ["flights_2015"] * len(df_month),
        "ds": pd.to_datetime(
            {
                "year": [2015] * len(df_month),
                "month": df_month["month"].astype(int),
                "day": [1] * len(df_month),
            }
        ),
        "y": df_month["total_flights"].astype(float),
    }
)

df_ts

In [ ]:
train = df_ts[df_ts["ds"] <= "2015-09-01"].copy()
test = df_ts[df_ts["ds"] > "2015-09-01"].copy()

display(train)
display(test)

In [ ]:
models = [
    AutoETS(season_length=12),
    AutoARIMA(season_length=12),
    AutoTheta(season_length=12),
]

sf = StatsForecast(models=models, freq="MS")
sf.fit(train)

forecast_df = sf.predict(h=9, level=[90]).reset_index()
forecast_df

In [ ]:
autoets_fitted = getattr(sf.fitted_[0, 0], "model_", sf.fitted_[0, 0])
autoarima_fitted = getattr(sf.fitted_[0, 1], "model_", sf.fitted_[0, 1])
autotheta_fitted = getattr(sf.fitted_[0, 2], "model_", sf.fitted_[0, 2])

print("AutoARIMA:", arima_string(autoarima_fitted))

if isinstance(autotheta_fitted, dict) and "modeltype" in autotheta_fitted:
    print("AutoTheta:", autotheta_fitted["modeltype"])
else:
    print("AutoTheta:", autotheta_fitted)

print("AutoETS fitted object:")
print(autoets_fitted)

In [ ]:
pred_test = forecast_df.iloc[:3].copy()
future_6 = forecast_df.iloc[3:].copy()

mae_df = pd.DataFrame(
    {
        "model": ["AutoETS", "AutoARIMA", "AutoTheta"],
        "mae": [
            mean_absolute_error(test["y"], pred_test["AutoETS"]),
            mean_absolute_error(test["y"], pred_test["AutoARIMA"]),
            mean_absolute_error(test["y"], pred_test["AutoTheta"]),
        ],
    }
).sort_values("mae")

mae_df

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train["ds"], train["y"], marker="o", label="Train")
plt.plot(test["ds"], test["y"], marker="o", label="Test real")

for model_name in ["AutoETS", "AutoARIMA", "AutoTheta"]:
    plt.plot(pred_test["ds"], pred_test[model_name], marker="o", label=model_name)
    plt.fill_between(
        pred_test["ds"],
        pred_test[f"{model_name}-lo-90"],
        pred_test[f"{model_name}-hi-90"],
        alpha=0.15,
    )

plt.title("Evaluación sobre test set (Oct-Dic 2015)")
plt.xlabel("Fecha")
plt.ylabel("Vuelos")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(df_ts["ds"], df_ts["y"], marker="o", label="Serie real 2015")

for model_name in ["AutoETS", "AutoARIMA", "AutoTheta"]:
    plt.plot(future_6["ds"], future_6[model_name], marker="o", label=f"{model_name} forecast")
    plt.fill_between(
        future_6["ds"],
        future_6[f"{model_name}-lo-90"],
        future_6[f"{model_name}-hi-90"],
        alpha=0.15,
    )

plt.title("Pronóstico Ene-Jun 2016")
plt.xlabel("Fecha")
plt.ylabel("Vuelos")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plot_bar(mae_df, "model", "mae", "Comparación de MAE por modelo", rotation=0)

### Interpretación sugerida

Anota aquí:

- qué modelo obtuvo el menor MAE
- cómo se ven los intervalos
- qué tan confiable parece el pronóstico con solo un año de datos